In [1]:
import pandas as pd 
df = pd.read_csv('../data/processed/modelA_dataset.csv')

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   dyad                      1440 non-null   str    
 1   MonthYear                 1440 non-null   int64  
 2   event_count               1440 non-null   float64
 3   goldstein_std             1440 non-null   float64
 4   goldstein_min             1440 non-null   float64
 5   num_mentions_sum          1440 non-null   float64
 6   num_articles_sum          1440 non-null   float64
 7   num_sources_sum           1440 non-null   float64
 8   high_conflict_count       1440 non-null   float64
 9   low_conflict_count        1440 non-null   float64
 10  quad4_count               1440 non-null   float64
 11  high_conflict_pct         1440 non-null   float64
 12  low_conflict_pct          1440 non-null   float64
 13  quad4_pct                 1440 non-null   float64
 14  event_count_lag1   

In [8]:
# define data
X = df.loc[:, ['event_count_lag1', 'goldstein_std_lag1', 'goldstein_min_lag1', 'num_mentions_sum_lag1',
               'num_articles_sum_lag1', 'num_sources_sum_lag1', 'high_conflict_count_lag1', 'low_conflict_count_lag1',
               'quad4_count_lag1', 'high_conflict_pct_lag1', 'low_conflict_pct_lag1', 'quad4_pct_lag1']]
y = df['monthly_label']

# 先按時間一刀切
train_mask = df['MonthYear'] < 202301
test_mask =  df['MonthYear'] >= 202301
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

In [13]:
# lgb model
from lightgbm import LGBMClassifier
from sklearn import set_config
set_config(display='text')
lgb_params = {
    'objective': 'multiclass',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,       
    'num_leaves': 7,             
    'max_depth': 3,               
    'min_child_samples': 20,      
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,             
    'reg_lambda': 0.1,            
    'random_state': 42,
    'n_jobs': -1,
    'verbose': 1,
}
mod1 = LGBMClassifier(**lgb_params)
mod1.fit(X_train, y_train)



[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.034566 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2813
[LightGBM] [Info] Number of data points in the train set: 1044, number of used features: 12
[LightGBM] [Info] Start training from score -0.295374
[LightGBM] [Info] Start training from score -2.396938
[LightGBM] [Info] Start training from score -1.803320
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_jobs=-1,
               num_leaves=7, objective='multiclass', random_state=42,
               reg_alpha=0.1, reg_lambda=0.1, subsample=0.8, verbose=1)

In [21]:
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
predict_train = mod1.predict(X_train)
predict_test = mod1.predict(X_test)
report_train = classification_report(y_train, predict_train)
report_test = classification_report(y_test, predict_test)
print(f'train: {report_train}')
print(f'test: {report_test}')


train:                precision    recall  f1-score   support

  Cooperation       0.91      0.96      0.93       777
High_Conflict       0.81      0.74      0.77        95
 Low_Conflict       0.79      0.62      0.69       172

     accuracy                           0.88      1044
    macro avg       0.84      0.77      0.80      1044
 weighted avg       0.88      0.88      0.88      1044

test:                precision    recall  f1-score   support

  Cooperation       0.88      0.95      0.92       244
High_Conflict       0.62      0.52      0.57        44
 Low_Conflict       0.76      0.67      0.71       108

     accuracy                           0.83       396
    macro avg       0.75      0.71      0.73       396
 weighted avg       0.82      0.83      0.82       396



In [ ]:
y_pred_proba = mod1.predict_proba(X_test)  
auc_macro = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
print(f'Macro AUC: {auc_macro:.3f}')

#　個別
from sklearn.preprocessing import label_binarize
classes = mod1.classes_
y_test_binarized = label_binarize(y_test, classes=classes)
for i, cls in enumerate(classes):
    auc = roc_auc_score(y_test_binarized[:, i], y_pred_proba[:, i])
    print(f'{cls} AUC: {auc:.3f}')


Macro AUC: 0.920
Cooperation AUC: 0.941
High_Conflict AUC: 0.939
Low_Conflict AUC: 0.879


In [15]:
# lgb model(嘗試進行倒數加權檢驗 )
from lightgbm import LGBMClassifier
from sklearn import set_config
from sklearn.utils.class_weight import compute_sample_weight

set_config(display='text')
lgb_params = {
    'objective': 'multiclass',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,       
    'num_leaves': 7,             
    'max_depth': 3,               
    'min_child_samples': 20,      
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,             
    'reg_lambda': 0.1,            
    'random_state': 42,
    'n_jobs': -1,
    'verbose': 1,
}
mod2 = LGBMClassifier(**lgb_params)
mod2.fit(X_train, y_train,
         sample_weight=compute_sample_weight('balanced',  y_train)) 


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001462 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2813
[LightGBM] [Info] Number of data points in the train set: 1044, number of used features: 12
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_jobs=-1,
               num_leaves=7, objective='multiclass', random_state=42,
               reg_alpha=0.1, reg_lambda=0.1, subsample=0.8, verbose=1)

In [16]:
from sklearn.metrics import classification_report
predict_train = mod2.predict(X_train)
predict_test = mod2.predict(X_test)
report_train = classification_report(y_train, predict_train)
report_test = classification_report(y_test, predict_test)
print(f'train: {report_train}')
print(f'test: {report_test}')


train:                precision    recall  f1-score   support

  Cooperation       0.96      0.84      0.90       777
High_Conflict       0.62      0.93      0.74        95
 Low_Conflict       0.62      0.80      0.70       172

     accuracy                           0.84      1044
    macro avg       0.73      0.85      0.78      1044
 weighted avg       0.87      0.84      0.85      1044

test:                precision    recall  f1-score   support

  Cooperation       0.94      0.88      0.91       244
High_Conflict       0.59      0.75      0.66        44
 Low_Conflict       0.69      0.71      0.70       108

     accuracy                           0.82       396
    macro avg       0.74      0.78      0.76       396
 weighted avg       0.83      0.82      0.83       396



In [23]:
# mod2 auc
y_pred_proba = mod2.predict_proba(X_test)  
auc_macro = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
print(f'Macro AUC: {auc_macro:.3f}')

#　個別
from sklearn.preprocessing import label_binarize
classes = mod2.classes_
y_test_binarized = label_binarize(y_test, classes=classes)
for i, cls in enumerate(classes):
    auc = roc_auc_score(y_test_binarized[:, i], y_pred_proba[:, i])
    print(f'{cls} AUC: {auc:.3f}')


Macro AUC: 0.911
Cooperation AUC: 0.944
High_Conflict AUC: 0.928
Low_Conflict AUC: 0.860


In [17]:
# lgb model(嘗試進行倒數勘根號加權檢驗 )
from lightgbm import LGBMClassifier
from sklearn import set_config
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

set_config(display='text')
lgb_params = {
    'objective': 'multiclass',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,       
    'num_leaves': 7,             
    'max_depth': 3,               
    'min_child_samples': 20,      
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,             
    'reg_lambda': 0.1,            
    'random_state': 42,
    'n_jobs': -1,
    'verbose': 1,
}
mod3 = LGBMClassifier(**lgb_params)
mod3.fit(X_train, y_train,
         sample_weight=np.sqrt(compute_sample_weight('balanced',  y_train))) 


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000129 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2813
[LightGBM] [Info] Number of data points in the train set: 1044, number of used features: 12
[LightGBM] [Info] Start training from score -0.598923
[LightGBM] [Info] Start training from score -1.649705
[LightGBM] [Info] Start training from score -1.352896
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_jobs=-1,
               num_leaves=7, objective='multiclass', random_state=42,
               reg_alpha=0.1, reg_lambda=0.1, subsample=0.8, verbose=1)

In [18]:
from sklearn.metrics import classification_report
predict_train = mod3.predict(X_train)
predict_test = mod3.predict(X_test)
report_train = classification_report(y_train, predict_train)
report_test = classification_report(y_test, predict_test)
print(f'train: {report_train}')
print(f'test: {report_test}')


train:                precision    recall  f1-score   support

  Cooperation       0.94      0.92      0.93       777
High_Conflict       0.77      0.91      0.83        95
 Low_Conflict       0.71      0.72      0.71       172

     accuracy                           0.88      1044
    macro avg       0.81      0.85      0.82      1044
 weighted avg       0.89      0.88      0.88      1044

test:                precision    recall  f1-score   support

  Cooperation       0.90      0.93      0.91       244
High_Conflict       0.55      0.59      0.57        44
 Low_Conflict       0.73      0.67      0.70       108

     accuracy                           0.82       396
    macro avg       0.73      0.73      0.73       396
 weighted avg       0.82      0.82      0.82       396



In [24]:
# auc mod3
y_pred_proba = mod3.predict_proba(X_test)  
auc_macro = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
print(f'Macro AUC: {auc_macro:.3f}')

#　個別
from sklearn.preprocessing import label_binarize
classes = mod3.classes_
y_test_binarized = label_binarize(y_test, classes=classes)
for i, cls in enumerate(classes):
    auc = roc_auc_score(y_test_binarized[:, i], y_pred_proba[:, i])
    print(f'{cls} AUC: {auc:.3f}')


Macro AUC: 0.917
Cooperation AUC: 0.944
High_Conflict AUC: 0.935
Low_Conflict AUC: 0.873


### 紀錄
### 模型比較與調參過程(此模型尚未加入政體差異的變數)
 
**Classfication Report**

| 版本 | H_C Recall | L_C Recall | Precision | F1 | Test Accuracy | 訓練/測試落差 |
|---|---|---|---|---|---|---|
| LGBM 原始參數 | 0.36 | 0.59 | 0.53 | 0.43 | 0.79 | 大過擬合 |
| LGBM 調參 | 0.52 | 0.67 | 0.62 | 0.57 | 0.83 | 縮小至0.05，過擬合大幅改善 |
| LGBM 調參 + 完全加權 | **0.75** | **0.71** | 0.59 | **0.66** | 0.82 | 穩定 |
| LGBM 調參 + 開根號加權 | 0.59 | 0.67 | 0.55 | 0.57 | 0.82 | 穩定，但 F1 未優於完全加權 |

**AUC**

| 版本 | Macro AUC | Cooperation AUC | High_Conflict AUC | Low_Conflict AUC |
|---|---|---|---|---|
| LGBM 原始參數 | 0.920 | 0.941 | 0.939 | 0.879 |
| LGBM 調參 + sample_weight 開根號加權 | 0.911 | 0.944 | 0.928 | 0.860 |
| LGBM 調參 + sample_weight balanced 完全加權（最終選定） | 0.917 | 0.944 | 0.935 | 0.873 |
 
**資料切分邏輯** : 時間切分（非隨機切分，因資料具時間序列與 panel 特性），以 2023-01 為界，之前為訓練集（1,044 筆），之後為測試集（396 筆）。
之後可嘗試Walk-Forward Validation

**加權策略比較**：原先預期開根號加權會在「不加權」與「完全加權」間取得更好平衡，
但實測結果 F1 與不加權版本打平，兩者皆低於完全加權版本，顯示對此資料集而言，
直接使用類別頻率完全反比的加權方式即為目前測試範圍內的最佳選擇，
而非越保守/越精緻的調整必然越好。

**關鍵發現**：三個版本的 AUC 差距極小（Macro AUC 皆落在 0.91-0.92），
但 recall/precision 在不同加權策略下差距很大（如 High_Conflict recall 從 0.36 到 0.75）。
這代表**模型本身區分「像不像 High_Conflict」的排序能力，從頭到尾都相當穩定且良好**，
class_weight／sample_weight 加權策略改變的主要是「最終判斷門檻的敏感度」，
而非模型底層的學習能力。也就是說，問題不在模型學得好不好（AUC 已證實學得不錯），
而在於「用什麼門檻把機率轉換成最終標籤」這個決策層面的選擇——
這代表未來可透過 threshold tuning（調整判定門檻），
在不重新訓練模型的前提下，於 recall／precision 間找到更精確的平衡點。或者未來預測改採報機率的方式，而不強制進行分類，機率輸出可避免單一門檻選擇造成的資訊損失，讓使用者自行判讀風險程度。

**註** : 這個調餐只是先大致修正預設參數，之後將透過Optuna進行更精細的調餐